## 1. Manual Constraint Solving
- Let's solve a small constraint satisfication problem by hand to experience how structured reasoning works.
- You are organizing a conference with four parallel workshops: Ethics in AI, NLP, Planning, Robotics
- There are four guest speakers: Dr. Lin, Prof. Silva, Dr. Morales, Dr. Kwon
- Each workshop must have exactly one speaker, and each speaker can only be assigned to one workshop.
- Constraints: 
    - Dr. Lin cannot speak in Planning.
    - Dr. Lin previously attended NLP.
    - Dr. Morales must speak in either NLP or Ethics.
    - Dr. Kwon previously attended Ethics.
    - Prof. Silva prefers Robotics but may speak in Planning if needed.
    
(a) Submit a valid assignment of speakers to workshops. 

(b) Briefly explain how you reasoned through the constraints. What strategies did you use? Did you eliminate values first? Did you backtrack? 

### Response

- I modeled this as a CSP with one variable per speaker and one value per workshop:
$$
\text{Workshops} = \{E, N, P, R\}
$$
where $E=$ Ethics in AI, $N=$ NLP, $P=$ Planning, $R=$ Robotics.

- Let variables be $L$ (Dr. Lin), $S$ (Prof. Silva), $M$ (Dr. Morales), $K$ (Dr. Kwon), with an all-different constraint:
$$
L, S, M, K \text{ must be all different.}
$$

- From the prompt constraints, I used these domains:
$$
L \in \{E, R\} \quad (L \neq P,\; L \neq N),
$$
$$
M \in \{N, E\},
$$
$$
K \in \{N, P, R\} \quad (K \neq E),
$$
$$
S \in \{R, P\} \text{ (prefers Robotics; Planning is acceptable if needed).}
$$
- Using notation from AIMA we can write the preference constraint as: 
$S : R \succ P$

- Now I reasoned by elimination with domain pruning / forward checking in mind:

1. Choose $M=E$ (allowed by $M \in \{N,E\}$).
2. Then $L \neq E$, so $L=R$.
3. Since $L=R$, Silva cannot take $R$, so $S=P$.
4. Remaining workshop is $N$, and $K$ can take $N$, so $K=N$.

- This gives a valid assignment:
$$
\boxed{\text{Ethics} \to \text{Dr. Morales},\; \text{NLP} \to \text{Dr. Kwon},\; \text{Planning} \to \text{Prof. Silva},\; \text{Robotics} \to \text{Dr. Lin}.}
$$

- I did not need full backtracking here because each step quickly forced the next value once I fixed $M=E$. The strategy was:
    - encode domains from constraints,
    - enforce all-different,
    - pick a constrained variable/value,
    - propagate eliminations until all variables are assigned.

I followed a simialar workflow shown in the lectures and course material: represent variables/domains/constraints, then reduce legal values via consistency checks and assignment. In `csp.py` from aima code, this is the same overall idea used by `nconflicts`, `forward_checking`, and `backtracking_search`.

## 2. Reformulating the N-Queens Problem as Algebraic Constraints (Lecture 6A)

A CSP can be reformulated as a system of equations, where satisfying all constraints corresponds to minimizing a global violation function.

You will practice this reformulation step below using a reduced version of the N-Queens problem with 3 queens.

**Problem Definition:** Place 3 queens on a 3×3 board such that:

- Exactly one queen per row  
- Exactly one queen per column  
- At most one queen per diagonal  

Let $Q_{i,j} \in \{0,1\}$, where $Q_{i,j} = 1$ indicates a queen at row $i$, column $j$, and $Q_{i,j} = 0$ otherwise.

(a) Clearly define the variables, domains, and constraints.

(b) Write the row constraints as equations.

(c) Write the column constraints as equations.

(d) Write one diagonal constraint explicitly (for example, the diagonal containing $Q_{1,1}, Q_{2,2}, Q_{3,3}$)

(e) Rewrite one row and one column constraint as squared violation terms.

(f) Construct a symbolic total violation function: $E = \sum (\text{constraint violation})^2$

(g) Explain why does $E = 0$ correspond to a feasible solution?

(h) How does this differ from solving the CSP via backtracking?

(i) Is this closer to logical reasoning, search, or optimization? Explain briefly.

### Response

**(a) Variables, domains, constraints**

- Variables: $Q_{i,j}$ for $i,j \in \{1,2,3\}$.
- Domain: $Q_{i,j} \in \{0,1\}$, where $1$ means a queen is placed at $(i,j)$.
- Constraints:
  - Exactly one queen in each row.
  - Exactly one queen in each column.
  - At most one queen on each diagonal (both main and anti-diagonals).
  - This means we also want to optimize it such that there is a queen in each row and column but not necesarilly every diagonal.

**(b) Row constraints as equations**
$$
\begin{aligned}
Q_{1,1}+Q_{1,2}+Q_{1,3} &= 1,\\
Q_{2,1}+Q_{2,2}+Q_{2,3} &= 1,\\
Q_{3,1}+Q_{3,2}+Q_{3,3} &= 1.
\end{aligned}
$$

**(c) Column constraints as equations**
$$
\begin{aligned}
Q_{1,1}+Q_{2,1}+Q_{3,1} &= 1,\\
Q_{1,2}+Q_{2,2}+Q_{3,2} &= 1,\\
Q_{1,3}+Q_{2,3}+Q_{3,3} &= 1.
\end{aligned}
$$

**(d) One diagonal constraint**

- For the main diagonal $(1,1),(2,2),(3,3)$:
$$
Q_{1,1}+Q_{2,2}+Q_{3,3} \le 1.
$$

**(e) One row and one column as squared violation terms**

- The idea here is to convert an exact-equality constraint into a nonnegative penalty that is $0$ only when the constraint is satisfied.  
- If a row or column must sum to $1$, then any deviation from $1$ is an error. Squaring the error gives a smooth penalty and prevents negatives.

So for one row and one column, the violation terms are:
$$
v_{r1} = (Q_{1,1}+Q_{1,2}+Q_{1,3}-1)^2
$$
$$
v_{c1} = (Q_{1,1}+Q_{2,1}+Q_{3,1}-1)^2
$$
- If row 1 has exactly one queen, then $v_{r1}=0$; if it has $0$ or $2$ queens, then $v_{r1}=1$; if it has $3$ queens, then $v_{r1}=4$. The same interpretation applies to $v_{c1}$.

**(f) Symbolic total violation function**

- Next, I aggregate all individual penalties into one global objective $E(Q)$. Minimizing $E$ means searching for assignments that jointly satisfy every constraint.

- I combine row/column equality penalties with diagonal overfill penalties (only penalize when a diagonal has more than one queen and not if it has none):
$$
\begin{aligned}
E(Q) &= \sum_{i=1}^{3}\left(\sum_{j=1}^{3}Q_{i,j}-1\right)^2
     + \sum_{j=1}^{3}\left(\sum_{i=1}^{3}Q_{i,j}-1\right)^2 \\
&\quad + 1/2*\sum_{d\in \mathcal{D}_{\text{main}}}\left(\max\left(0,\sum_{(i,j)\in d}Q_{i,j}-1\right)\right)^2
     + 1/2*\sum_{d\in \mathcal{D}_{\text{anti}}}\left(\max\left(0,\sum_{(i,j)\in d}Q_{i,j}-1\right)\right)^2.
\end{aligned}
$$
- This is the same modeling pattern as the energy-based CSP framing in the Q'tron paper: define nonnegative sub-energies for row, column, and both diagonal families, then sum them into a total energy. In their N-Queens construction, the total energy is written as $E_{n\text{-queen}} = E_{-} + E_{|} + E_{\\} + E_{/}$ (Section 4.2, Eqs. (29)-(33)).
- For the diagonols they are multiplied by $1/2$ to prevent ordered pairs from being duplicated. The max modifier is used to pick 0 penalty if we only have 1 queen and the penalty if there is more than 1 queen on a diagonol.

**(g) Why does $E=0$ mean feasible?**

- Every term in $E$ is a square, so each term is nonnegative.
- Therefore, $E=0$ can happen only if every individual violation term is $0$.
- That means every row/column equality is satisfied exactly, and no diagonal exceeds one queen, so all constraints are satisfied.

**(h) How is this different from backtracking?**

- Backtracking treats CSP solving as a discrete search over partial assignments, pruning inconsistent branches early (as in `backtracking_search`, `forward_checking`, and `nconflicts` in `csp.py` aima code) using methods like forward checking local neighbors.
- The algebraic form treats it as minimizing an objective $E$; constraint satisfaction appears as the special case where the minimum reaches $0$. The goal is to minimize the energy topology of the system. When it is minimized is when our CSP is satisfied.

**(i) Is this reasoning, search, or optimization?**

- Using systems of equations and formulating a function to minimzie is closest to optimization.
- Logical constraints are encoded into a numeric energy/penalty and optimized, rather than directly proving consistency step-by-step.

- Also, standard $3$-queens has no feasible non-attacking placement, so under full row/column/diagonal constraints the minimum achievable value is $E>0$ (not $0$). Every pattern permutation creates a diagonal attack.

**Reference used**

- Yue, T.-W., & Chen, M.-C. (2004). *Q'tron Neural Networks for Constraint Satisfaction*. See Section 3.4 (energy function form) and Section 4.2, Eqs. (29)-(33) (N-Queens energy decomposition).

## 3. Forward Checking Simulation

Consider the CSP: Color three regions (A, B, C) with colors {Red, Green} such that:

- $A \ne B$
- $B \ne C$

Initial domains are $\{R, G\}$ for each.

We will apply Backtracking with Forward Checking.

Simulate the algorithm step by step under the following ordering and submit the step-by-step trace.

- Variable order: $A \rightarrow B \rightarrow C$
- Value order: R before G

For each assignment attempt:

(a) Write the current assignment.

(b) Show the updated domains after Forward Checking.

(c) Indicate whether continue, backtrack, or solution found.

### Response
- Initial domains:
$$
D(A)=D(B)=D(C)=\{R,G\}
$$

1. Assign $A=R$.
   - Current assignment: $\{A=R\}$
   - Forward checking on neighbor $B$ (constraint $A \ne B$): remove $R$ from $D(B)$.
   - Updated domains: $D(B)=\{G\}$, $D(C)=\{R,G\}$
   - Result = continue.

2. Try $B=R$ first (based on topological order).
   - This is immediately inconsistent with $A \ne B$ since $A=R$.
   - No updates to the domains because no forward-checing update is made due to value rejection.
   - Result = try next domain value

3. Assign $B=G$.
   - Current assignment: $\{A=R, B=G\}$
   - Forward checking on neighbor $C$ (constraint $B \ne C$): remove $G$ from $D(C)$.
   - Updated domains: $D(C)=\{R\}$
   - Result = continue.

4. Assign $C=R$ (only domain value left due to pruning).
   - Current assignment: $\{A=R, B=G, C=R\}$
   - All constraints satisfied:
     - $A \ne B$: $R \ne G$ true
     - $B \ne C$: $G \ne R$ true
   - Result = solution found.

- Final solution:
$$
\boxed{A=R,\; B=G,\; C=R}
$$

- No backtracking to a previous variable was needed because forward checking pruned domains enough to make the remaining choices deterministic.